In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import torch
import torch.nn as nn
import numpy as np
import math

# Force float64 for higher precision in ODE derivatives
torch.set_default_dtype(torch.float64)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------
# 1. Model Components
# ----------------------------
class FourierFeatures(nn.Module):
    def __init__(self, in_dim=2, mapping_size=128, sigma=1.0):
        super().__init__()
        # B matrix is [in_dim, mapping_size]
        # This projects our 2D input into a higher-dimensional space
        self.register_buffer('B', torch.randn(in_dim, mapping_size) * sigma)

    def forward(self, x):
        # x: [N, 2] @ B: [2, mapping_size] -> proj: [N, mapping_size]
        proj = 2.0 * math.pi * (x @ self.B)
        # Concatenate sin and cos -> Output size: [N, 2 * mapping_size]
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)

class PINN(nn.Module):
    def __init__(self, in_dim=2, width=64, mapping_size=128, depth=4, out_dim=1, sigma=1.0, x0=0.7, v0=1.2):
        super().__init__()
        self.x0, self.v0 = x0, v0
        
        # 1. Fourier Layer
        # Input: [N, 2] -> Output: [N, 2 * mapping_size]
        self.ff = FourierFeatures(in_dim, mapping_size=mapping_size, sigma=sigma)
        
        # 2. MLP Architecture
        layers = []
        
        # FIRST LAYER: Connects Fourier output to the MLP width
        # The input dimension is (2 * mapping_size) because of the sin/cos concat
        layers.append(nn.Linear(2 * mapping_size, width))
        layers.append(nn.Tanh())
        
        # HIDDEN LAYERS: width -> width
        for _ in range(depth - 1):
            layers.append(nn.Linear(width, width))
            layers.append(nn.Tanh())
            
        # OUTPUT LAYER: width -> 1
        layers.append(nn.Linear(width, out_dim))
        
        self.net = nn.Sequential(*layers)

    def forward(self, z):
        # z is [N, 2] -> [t, xi]
        t = z[:, 0:1]
        
        # Pass inputs through Fourier mapping, then MLP
        ff_out = self.ff(z)
        N_out = self.net(ff_out)
        
        # Hard Ansatz: x(t) = x0 + v0*t + t^2 * N(t, xi)
        # Automatically satisfies x(0) = x0 and x'(0) = v0
        gate = (1.0 - torch.exp(-t))**2
        
        phi = self.v0 * (1.0 - torch.exp(-t))
        # 4. Neural Network Residual
        N = N_out * (1.0 - torch.exp(-t))**2
        
        return self.x0 + phi + gate * N
# ----------------------------
# 2. Physics & Exact Solution
# ----------------------------
def get_exact_solution_torch(t, xi, x0=0.7, v0=1.2):
    wd = torch.sqrt(torch.clamp(1.0 - xi**2, min=1e-9))
    A = x0
    B = (v0 + xi * x0) / wd
    return torch.exp(-xi * t) * (A * torch.cos(wd * t) + B * torch.sin(wd * t))

def ode_loss(model, z):
    z.requires_grad_(True)
    x = model(z)
    
    # Get dx/dt
    grads = torch.autograd.grad(x, z, torch.ones_like(x), create_graph=True)[0]
    dx_dt = grads[:, 0:1]
    
    # Get d^2x/dt^2
    grads2 = torch.autograd.grad(dx_dt, z, torch.ones_like(dx_dt), create_graph=True)[0]
    ddx_dt2 = grads2[:, 0:1]
    
    xi = z[:, 1:2]
    
    # Equation: x'' + 2*xi*x' + x = 0
    residual = ddx_dt2 + 2 * xi * dx_dt + x
    return torch.mean(residual**2)

def sample_interior_points(N, T, device):
    t = torch.rand(N, 1, device=device) * T
    xi = 0.1 + (0.4 - 0.1) * torch.rand(N, 1, device=device)
    return torch.cat([t, xi], dim=1)



In [ ]:
# ----------------------------
# 3. Training Loop
# ----------------------------
T = 20.0
sigma = 2.0 # Natural frequency roughly ~ 1
num_samples = 7500
seed = 0

# Exponential loop for Adam Learning Rates (e.g., 0.01 to 0.0001)
learning_rates = np.logspace(-6, -4, num=3)

print(f"\n{'='*30} Testing T = {T} | Sigma = {sigma} {'='*30}")

for lr in learning_rates:
    print(f"\n>> Configuration: Adam LR = {lr:.4f}")
    
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    z_i = sample_interior_points(num_samples, T, device).requires_grad_(True)
    model = PINN(in_dim=2, sigma=sigma).to(device)

    # --- Phase 1: Adam Optimization ---
    optimizer_adam = torch.optim.Adam(model.parameters(), lr=lr)
    for step in range(3000):
        optimizer_adam.zero_grad()
        loss = ode_loss(model, z_i)
        loss.backward()
        optimizer_adam.step()
        
        if step % 100 == 0:
            print(f"   Adam Step {step:4d} | Loss: {loss.item():.4e}")

    adam_err = ode_loss(model, z_i).item()

    # --- Phase 2: L-BFGS Optimization ---
    optimizer_lbfgs = torch.optim.LBFGS(
        model.parameters(), lr=1.0, max_iter=1600, history_size=100, 
        tolerance_grad=1e-15, tolerance_change=1e-15, line_search_fn="strong_wolfe"
    )
    
    def closure():
        optimizer_lbfgs.zero_grad()
        l = ode_loss(model, z_i)
        l.backward()
        return l
        
    optimizer_lbfgs.step(closure)
    final_err = ode_loss(model, z_i).item()
    
    # --- Global L2 Evaluation ---
    model.eval()
    with torch.no_grad():
        t_axis = torch.linspace(0, T, 100, device=device)
        xi_axis = torch.linspace(0.1, 0.4, 30, device=device)
        z_test = torch.cartesian_prod(t_axis, xi_axis)
        
        x_pred = model(z_test) 
        x_true = get_exact_solution_torch(z_test[:, 0:1], z_test[:, 1:2])
    
        rel_l2 = torch.norm(x_true - x_pred) / torch.norm(x_true)
        
    model.train()

    # Final logs for this learning rate
    print(f"   End of Adam Error  : {adam_err:.4e}")
    print(f"   Final L-BFGS Error : {final_err:.4e}")
    print(f"   Relative L2 Error  : {rel_l2.item():.4e}")


============================== Testing T = 20.0 | Sigma = 2.0 ==============================

>> Configuration: Adam LR = 0.0000
   Adam Step    0 | Loss: 1.0778e+02
   Adam Step  100 | Loss: 9.5974e+01
   Adam Step  200 | Loss: 8.5654e+01
   Adam Step  300 | Loss: 7.6617e+01
   Adam Step  400 | Loss: 6.8682e+01
   Adam Step  500 | Loss: 6.1700e+01
   Adam Step  600 | Loss: 5.5546e+01
   Adam Step  700 | Loss: 5.0114e+01
   Adam Step  800 | Loss: 4.5318e+01
   Adam Step  900 | Loss: 4.1079e+01
   Adam Step 1000 | Loss: 3.7334e+01
   Adam Step 1100 | Loss: 3.4024e+01
   Adam Step 1200 | Loss: 3.1098e+01
   Adam Step 1300 | Loss: 2.8511e+01
   Adam Step 1400 | Loss: 2.6222e+01
   Adam Step 1500 | Loss: 2.4195e+01
   Adam Step 1600 | Loss: 2.2399e+01
   Adam Step 1700 | Loss: 2.0804e+01
   Adam Step 1800 | Loss: 1.9385e+01
   Adam Step 1900 | Loss: 1.8121e+01
   Adam Step 2000 | Loss: 1.6990e+01
   Adam Step 2100 | Loss: 1.5978e+01
   Adam Step 2200 | Loss: 1.5067e+01
   Adam Step 2300 |

In [ ]:
import matplotlib.pyplot as plt

t_axis = torch.linspace(0, T, 100, device=device)
xi_axis = torch.linspace(0.1, 0.4, 30, device=device)

Tg, Xig = torch.meshgrid(t_axis, xi_axis, indexing="ij")

x_pred_grid = x_pred.reshape(100,30).cpu().numpy()
x_true_grid = x_true.reshape(100,30).cpu().numpy()

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.contourf(Tg.cpu(), Xig.cpu(), x_true_grid, levels=50)
plt.title("Exact Solution")
plt.xlabel("t")
plt.ylabel("xi")
plt.colorbar()

plt.subplot(1,2,2)
plt.contourf(Tg.cpu(), Xig.cpu(), x_pred_grid, levels=50)
plt.title("PINN Prediction")
plt.xlabel("t")
plt.ylabel("xi")
plt.colorbar()

plt.tight_layout()
plt.show()

In [ ]:
error = np.abs(x_true_grid - x_pred_grid)

plt.figure(figsize=(6,5))
plt.contourf(Tg.cpu(), Xig.cpu(), error, levels=50, cmap='inferno')
plt.title("Absolute Error |u_true - u_pred|")
plt.xlabel("t")
plt.ylabel("xi")
plt.colorbar()
plt.show()

In [ ]:
xi_index = 15  # mid xi slice

plt.figure(figsize=(7,5))
plt.plot(t_axis.cpu(), x_true_grid[:,xi_index], label="Exact")
plt.plot(t_axis.cpu(), x_pred_grid[:,xi_index], '--', label="PINN")
plt.xlabel("t")
plt.ylabel("u(t,xi)")
plt.title(f"Solution slice at xi={xi_axis[xi_index]:.3f}")
plt.legend()
plt.show()

In [ ]:
rel_error_t = np.linalg.norm(x_true_grid - x_pred_grid, axis=1) / np.linalg.norm(x_true_grid, axis=1)

plt.plot(t_axis.cpu(), rel_error_t)
plt.xlabel("t")
plt.ylabel("Relative L2 Error")
plt.title("Relative Error vs Time")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# ----- Dense evaluation grid -----
t_axis = torch.linspace(0, T, 150, device=device)
xi_axis = torch.linspace(0.1, 0.4, 80, device=device)

Tg, Xig = torch.meshgrid(t_axis, xi_axis, indexing="ij")
z_test = torch.stack([Tg.reshape(-1), Xig.reshape(-1)], dim=1)

# ----- Evaluate model and exact solution -----
model.eval()
with torch.no_grad():
    x_pred = model(z_test).reshape(150, 80)
    x_true = get_exact_solution_torch(
        z_test[:,0:1], z_test[:,1:2]
    ).reshape(150,80)

Tg = Tg.cpu().numpy()
Xig = Xig.cpu().numpy()
x_pred = x_pred.cpu().numpy()
x_true = x_true.cpu().numpy()

# ----- Plot surfaces -----
fig = plt.figure(figsize=(14,6))

# Exact solution
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(Tg, Xig, x_true, cmap='viridis', linewidth=0)
ax1.set_title("Exact Solution")
ax1.set_xlabel("t")
ax1.set_ylabel("xi")
ax1.set_zlabel("u(t,xi)")

# PINN prediction
ax2 = fig.add_subplot(122, projection='3d')
ax2.plot_surface(Tg, Xig, x_pred, cmap='viridis', linewidth=0)
ax2.set_title("PINN Prediction")
ax2.set_xlabel("t")
ax2.set_ylabel("xi")
ax2.set_zlabel("u(t,xi)")

plt.tight_layout()
plt.show()